# Imbalance experiments: CatBoost road-only 3.5

Цель ноутбука — проверить, насколько текущие ложные срабатывания связаны с дисбалансом классов и выбранной стратегией `auto_class_weights="Balanced"`.

Итог работы ноутбука — сводные CSV-таблицы в `reports/`, которые можно загрузить обратно в чат и разобрать отдельно:

- `reports/catboost_road_only_3_5_imbalance_summary.csv` — одна строка на эксперимент;
- `reports/catboost_road_only_3_5_imbalance_thresholds.csv` — метрики по разным threshold для каждого эксперимента;
- `reports/catboost_road_only_3_5_imbalance_calibration.csv` — calibration bins для вероятностной интерпретации score.

## 0. Что проверяем

Базовая проблема: положительный класс редкий, поэтому модель может либо слишком часто говорить `0`, либо при усилении положительного класса ловить много ДТП, но давать много ложных срабатываний.

Проверяем несколько стратегий:

1. `balanced` — текущий baseline: `auto_class_weights="Balanced"`.
2. `no_weights` — CatBoost без class weights.
3. `scale_pos_weight` — ручной вес положительного класса.
4. `undersampling` — берем все positive и случайную часть negative в train.
5. `threshold tuning` — смотрим, как меняются `TP/FP/FN/TN`, precision и recall при разных порогах.
6. `calibration` — проверяем, можно ли score трактовать как вероятность.

Focal loss отдельно не запускается: в CatBoost это не штатный простой параметр. Его лучше рассматривать позже, если обычные стратегии дисбаланса не дадут понимания.

## 1. Imports and config

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_accident_risk.catboost_baseline import (  # noqa: E402
    FEATURE_SET_ROAD_WITHOUT_TYPE_LENGTHS_NO_CALENDAR_SLIM,
    build_features,
)
from road_accident_risk.logreg_baseline import TARGET, TRAIN_END, precision_recall_at_k  # noqa: E402
from road_accident_risk.paths import H3_MONTH_ROAD_FEATURES_PARQUET  # noqa: E402

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = REPORTS_DIR / "catboost_road_only_3_5_imbalance_summary.csv"
THRESHOLDS_PATH = REPORTS_DIR / "catboost_road_only_3_5_imbalance_thresholds.csv"
CALIBRATION_PATH = REPORTS_DIR / "catboost_road_only_3_5_imbalance_calibration.csv"

RANDOM_SEED = 42
ITERATIONS = 800
LEARNING_RATE = 0.05
DEPTH = 6

# Calibration обучается только на 2022 годе, test остается 2023-2025.
CALIBRATION_START = "2022-01-01"
CALIBRATION_END = TRAIN_END

# Для первого быстрого прогона можно временно поставить меньше итераций.
# ITERATIONS = 300

## 2. Load road-only data

Используется тот же feature set, который принят для `CatBoost road-only 3.5`: `road_without_type_lengths_no_calendar_slim`.

In [ ]:
panel = pd.read_parquet(H3_MONTH_ROAD_FEATURES_PARQUET)
panel["month"] = pd.to_datetime(panel["month"])
panel[TARGET] = panel[TARGET].astype(int)

panel = panel[panel["road_length_total_m"] > 0].copy()
features = build_features(panel, feature_set=FEATURE_SET_ROAD_WITHOUT_TYPE_LENGTHS_NO_CALENDAR_SLIM)

train_full = panel[panel["month"] < TRAIN_END].copy()
test = panel[panel["month"] >= TRAIN_END].copy()

fit_train = train_full[train_full["month"] < CALIBRATION_START].copy()
calibration = train_full[(train_full["month"] >= CALIBRATION_START) & (train_full["month"] < CALIBRATION_END)].copy()

print(f"Panel rows: {len(panel):,}")
print(f"Feature count: {len(features):,}")
print(f"Train full: {len(train_full):,}, positive rate={train_full[TARGET].mean():.4%}")
print(f"Fit train for calibration: {len(fit_train):,}, positive rate={fit_train[TARGET].mean():.4%}")
print(f"Calibration: {len(calibration):,}, positive rate={calibration[TARGET].mean():.4%}")
print(f"Test: {len(test):,}, positive rate={test[TARGET].mean():.4%}")

## 3. Experiment grid

Каждый эксперимент меняет только стратегию работы с дисбалансом классов. Test set всегда полный, без undersampling.

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    strategy: str
    auto_class_weights: str | None = None
    scale_pos_weight: float | None = None
    negative_ratio: int | None = None
    calibrate: str | None = None  # None, "platt", "isotonic"


BASE_EXPERIMENTS = [
    ExperimentConfig(name="balanced", strategy="auto_class_weights", auto_class_weights="Balanced"),
    ExperimentConfig(name="no_weights", strategy="none"),
    ExperimentConfig(name="scale_pos_weight_1", strategy="scale_pos_weight", scale_pos_weight=1.0),
    ExperimentConfig(name="scale_pos_weight_3", strategy="scale_pos_weight", scale_pos_weight=3.0),
    ExperimentConfig(name="scale_pos_weight_5", strategy="scale_pos_weight", scale_pos_weight=5.0),
    ExperimentConfig(name="scale_pos_weight_10", strategy="scale_pos_weight", scale_pos_weight=10.0),
    ExperimentConfig(name="scale_pos_weight_20", strategy="scale_pos_weight", scale_pos_weight=20.0),
    ExperimentConfig(name="scale_pos_weight_40", strategy="scale_pos_weight", scale_pos_weight=40.0),
    ExperimentConfig(name="undersample_1_to_5", strategy="undersampling", negative_ratio=5),
    ExperimentConfig(name="undersample_1_to_10", strategy="undersampling", negative_ratio=10),
    ExperimentConfig(name="undersample_1_to_20", strategy="undersampling", negative_ratio=20),
]

CALIBRATION_EXPERIMENTS = [
    ExperimentConfig(name="balanced_platt", strategy="auto_class_weights", auto_class_weights="Balanced", calibrate="platt"),
    ExperimentConfig(name="balanced_isotonic", strategy="auto_class_weights", auto_class_weights="Balanced", calibrate="isotonic"),
]

EXPERIMENTS = BASE_EXPERIMENTS + CALIBRATION_EXPERIMENTS
pd.DataFrame([config.__dict__ for config in EXPERIMENTS])

## 4. Metric helpers

In [ ]:
def make_train_sample(df: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    if config.strategy != "undersampling":
        return df
    if config.negative_ratio is None:
        raise ValueError("negative_ratio is required for undersampling")

    positives = df[df[TARGET] == 1]
    negatives = df[df[TARGET] == 0]
    n_negatives = min(len(negatives), len(positives) * config.negative_ratio)
    sampled_negatives = negatives.sample(n=n_negatives, random_state=RANDOM_SEED)
    return pd.concat([positives, sampled_negatives], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)


def build_model(config: ExperimentConfig) -> CatBoostClassifier:
    params = {
        "iterations": ITERATIONS,
        "learning_rate": LEARNING_RATE,
        "depth": DEPTH,
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": RANDOM_SEED,
        "allow_writing_files": False,
        "verbose": 100,
    }
    if config.auto_class_weights is not None:
        params["auto_class_weights"] = config.auto_class_weights
    if config.scale_pos_weight is not None:
        params["scale_pos_weight"] = config.scale_pos_weight
    return CatBoostClassifier(**params)


def confusion_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> dict[str, float | int]:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    specificity = tn / (tn + fp) if tn + fp > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "precision_class_1": precision,
        "recall_class_1": recall,
        "specificity_class_0": specificity,
        "f1_class_1": f1,
        "predicted_positive_rate": float(y_pred.mean()),
    }


def threshold_sweep(y_true: np.ndarray, y_score: np.ndarray, thresholds: list[float]) -> pd.DataFrame:
    return pd.DataFrame([confusion_metrics(y_true, y_score, threshold) for threshold in thresholds])


def best_f1_threshold(thresholds_df: pd.DataFrame) -> dict[str, float | int]:
    row = thresholds_df.sort_values(["f1_class_1", "precision_class_1"], ascending=False).iloc[0]
    return row.to_dict()


def threshold_at_min_precision(thresholds_df: pd.DataFrame, min_precision: float) -> dict[str, float | int] | None:
    eligible = thresholds_df[thresholds_df["precision_class_1"] >= min_precision]
    if eligible.empty:
        return None
    return eligible.sort_values(["recall_class_1", "threshold"], ascending=[False, True]).iloc[0].to_dict()


def calibration_bins(y_true: np.ndarray, y_score: np.ndarray, experiment: str, n_bins: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"y_true": y_true, "score": y_score})
    df["bin"] = pd.qcut(df["score"], q=n_bins, duplicates="drop")
    out = df.groupby("bin", observed=True).agg(
        rows=("y_true", "size"),
        score_mean=("score", "mean"),
        observed_positive_rate=("y_true", "mean"),
    ).reset_index()
    out.insert(0, "experiment", experiment)
    out["bin"] = out["bin"].astype(str)
    return out

## 5. Training loop

Этот блок обучает все варианты и складывает результаты в одну таблицу.

Важно: calibration-эксперименты обучают CatBoost на `fit_train` до 2022 года, калибратор — на 2022 годе, а оцениваются на test 2023-2025. Некалиброванные эксперименты обучаются на полном train 2015-2022.

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

summary_rows = []
threshold_rows = []
calibration_rows = []
score_store = {}

X_test = test[features]
y_test = test[TARGET].to_numpy()

for config in EXPERIMENTS:
    print(f"\n=== {config.name} ===")
    train_source = fit_train if config.calibrate else train_full
    train_sample = make_train_sample(train_source, config)
    print(f"Train rows: {len(train_sample):,}, positive rate={train_sample[TARGET].mean():.4%}")

    model = build_model(config)
    train_pool = Pool(train_sample[features], train_sample[TARGET], feature_names=features)
    eval_pool = Pool(X_test, y_test, feature_names=features)
    model.fit(train_pool, eval_set=eval_pool, use_best_model=False)

    test_score = model.predict_proba(X_test)[:, 1]

    if config.calibrate:
        calibration_score = model.predict_proba(calibration[features])[:, 1]
        if config.calibrate == "platt":
            calibrator = LogisticRegression(solver="lbfgs", max_iter=1000)
            calibrator.fit(calibration_score.reshape(-1, 1), calibration[TARGET])
            test_score = calibrator.predict_proba(test_score.reshape(-1, 1))[:, 1]
        elif config.calibrate == "isotonic":
            calibrator = IsotonicRegression(out_of_bounds="clip")
            calibrator.fit(calibration_score, calibration[TARGET])
            test_score = calibrator.predict(test_score)
        else:
            raise ValueError(f"Unknown calibration method: {config.calibrate}")

    score_store[config.name] = test_score
    roc_auc = roc_auc_score(y_test, test_score)
    pr_auc = average_precision_score(y_test, test_score)
    brier = brier_score_loss(y_test, test_score)

    sweep = threshold_sweep(y_test, test_score, thresholds)
    sweep.insert(0, "experiment", config.name)
    threshold_rows.append(sweep)

    at_05 = sweep[sweep["threshold"] == 0.50].iloc[0].to_dict()
    best_f1 = best_f1_threshold(sweep)
    at_precision_20 = threshold_at_min_precision(sweep, min_precision=0.20)

    top_metrics = {}
    for k_frac in [0.01, 0.03, 0.05, 0.10]:
        metrics = precision_recall_at_k(y_test, test_score, k_frac=k_frac)
        key = f"top_{int(k_frac * 100)}pct"
        top_metrics[f"{key}_precision"] = metrics["precision"]
        top_metrics[f"{key}_recall"] = metrics["recall"]
        top_metrics[f"{key}_lift"] = metrics["lift"]
        top_metrics[f"{key}_tp"] = metrics["positives_in_top_k"]
        top_metrics[f"{key}_k"] = metrics["k"]

    summary_rows.append({
        "experiment": config.name,
        "strategy": config.strategy,
        "auto_class_weights": config.auto_class_weights,
        "scale_pos_weight": config.scale_pos_weight,
        "negative_ratio": config.negative_ratio,
        "calibrate": config.calibrate,
        "train_rows": len(train_sample),
        "train_positive_rate": train_sample[TARGET].mean(),
        "test_rows": len(test),
        "test_positive_rate": y_test.mean(),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "brier_score": brier,
        "threshold_05_tp": at_05["tp"],
        "threshold_05_fp": at_05["fp"],
        "threshold_05_fn": at_05["fn"],
        "threshold_05_tn": at_05["tn"],
        "threshold_05_precision": at_05["precision_class_1"],
        "threshold_05_recall": at_05["recall_class_1"],
        "threshold_05_f1": at_05["f1_class_1"],
        "best_f1_threshold": best_f1["threshold"],
        "best_f1_precision": best_f1["precision_class_1"],
        "best_f1_recall": best_f1["recall_class_1"],
        "best_f1": best_f1["f1_class_1"],
        "threshold_for_precision_20": None if at_precision_20 is None else at_precision_20["threshold"],
        "recall_at_precision_20": None if at_precision_20 is None else at_precision_20["recall_class_1"],
        **top_metrics,
    })

    calibration_rows.append(calibration_bins(y_test, test_score, experiment=config.name))
    print(f"ROC-AUC={roc_auc:.6f} PR-AUC={pr_auc:.6f} Brier={brier:.6f}")
    print(f"@0.5 precision={at_05['precision_class_1']:.4f} recall={at_05['recall_class_1']:.4f} fp={int(at_05['fp']):,} fn={int(at_05['fn']):,}")

## 6. Summary table

Главная таблица для последующего анализа. Ее нужно сохранить и загрузить в чат.

In [ ]:
summary = pd.DataFrame(summary_rows)
threshold_results = pd.concat(threshold_rows, ignore_index=True)
calibration_results = pd.concat(calibration_rows, ignore_index=True)

summary = summary.sort_values(["pr_auc", "top_1pct_precision"], ascending=False)
summary.to_csv(SUMMARY_PATH, index=False)
threshold_results.to_csv(THRESHOLDS_PATH, index=False)
calibration_results.to_csv(CALIBRATION_PATH, index=False)

print(SUMMARY_PATH)
print(THRESHOLDS_PATH)
print(CALIBRATION_PATH)

display_columns = [
    "experiment",
    "strategy",
    "calibrate",
    "scale_pos_weight",
    "negative_ratio",
    "roc_auc",
    "pr_auc",
    "brier_score",
    "threshold_05_precision",
    "threshold_05_recall",
    "threshold_05_fp",
    "threshold_05_fn",
    "top_1pct_precision",
    "top_5pct_precision",
    "best_f1_threshold",
    "best_f1",
]
summary[display_columns]

## 7. Threshold tuning table

Эта таблица нужна, чтобы смотреть компромисс `FP/FN`, `precision/recall` при разных порогах. Особенно полезно сравнивать `balanced`, `no_weights` и лучшие варианты `scale_pos_weight`.

In [ ]:
threshold_results.sort_values(["experiment", "threshold"]).head(30)

## 8. Calibration bins

Таблица показывает, насколько score похож на вероятность: `score_mean` сравнивается с фактической долей ДТП `observed_positive_rate` в каждом бине.

In [ ]:
calibration_results.head(30)

## 9. Optional quick plots

Графики не являются главным итогом. Главный итог — CSV summary. Но эти графики помогают быстро увидеть trade-off по threshold и calibration.

In [ ]:
import matplotlib.pyplot as plt

plot_experiments = ["balanced", "no_weights", "scale_pos_weight_5", "scale_pos_weight_10", "scale_pos_weight_20"]
plot_df = threshold_results[threshold_results["experiment"].isin(plot_experiments)]

plt.figure(figsize=(8, 5), dpi=130)
for experiment, part in plot_df.groupby("experiment"):
    plt.plot(part["recall_class_1"], part["precision_class_1"], marker="o", label=experiment)
plt.xlabel("Recall class 1")
plt.ylabel("Precision class 1")
plt.title("Threshold trade-off by imbalance strategy")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Что загрузить для анализа

После выполнения ноутбука загрузить сюда:

```text
reports/catboost_road_only_3_5_imbalance_summary.csv
```

Если захотим подробнее разобрать threshold tuning или calibration, дополнительно загрузить:

```text
reports/catboost_road_only_3_5_imbalance_thresholds.csv
reports/catboost_road_only_3_5_imbalance_calibration.csv
```